# 004 Workflows and Agents

这是 LangGraph 学习线的第四份 Notebook。

官方参考：

- https://docs.langchain.com/oss/python/langgraph/workflows-agents

学习目标：

1. 区分 workflow 和 agent 的控制权差异
2. 理解 prompt chaining、routing、parallelization、orchestrator-worker、evaluator-optimizer 等 workflow 模式
3. 跑通一个 evaluator-optimizer workflow
4. 跑通一个动态选择工具的 agent loop
5. 判断什么时候应该用 workflow，什么时候应该用 agent

这一讲继续使用 fake / deterministic 节点，不调用真实模型。

## 1. Workflow 和 Agent 的区别

可以先用一句话区分：

```text
Workflow：系统预先定义流程。
Agent：模型动态决定下一步。
```

更具体一点：

| 类型 | 下一步由谁决定 | 适合场景 |
| --- | --- | --- |
| Workflow | 系统代码 / graph edge | 流程稳定、标准清晰、需要可靠性 |
| Agent | 模型根据上下文决定 | 问题开放、步骤未知、需要探索 |

这不是谁更高级的问题。

核心判断是：

```text
如果流程稳定，就用 workflow。
如果步骤未知，才考虑 agent。
```

## 2. 常见 Workflow 模式

官方文档里常见的 workflow 模式可以这样理解：

| 模式 | 含义 | 例子 |
| --- | --- | --- |
| Prompt chaining | 一个步骤输出作为下个步骤输入 | 草稿 -> 改写 -> 总结 |
| Routing | 先分类，再走不同路径 | 售前 / 售后 / 财务 |
| Parallelization | 多个任务并行执行再合并 | 同时查 GitHub、Slack、Docs |
| Orchestrator-worker | 主流程拆任务给 worker | coordinator 分配 research 子任务 |
| Evaluator-optimizer | 生成、评估、改进循环 | 草拟回复 -> 评分 -> 重写 |

这节课先跑通 evaluator-optimizer，因为它最能体现 workflow 的稳定边界。

In [67]:
import operator
from typing import Annotated, Literal

from langchain_core.messages import AIMessage, AnyMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph
from typing_extensions import TypedDict


## 3. Workflow 示例：Evaluator-Optimizer

这个 workflow 处理客服回复草稿。

流程是固定的：

```text
draft_response -> evaluate_response
                   -> 如果不合格，回到 draft_response
                   -> 如果合格，进入 finalize_response
```

注意：

```text
是否重写由系统评估节点决定，
不是模型随便决定下一步。
```

In [68]:
class ReplyWorkflowState(TypedDict):
    user_request: str
    draft: str
    feedback: str
    quality: str
    iterations: int
    final_answer: str


def draft_response(state: ReplyWorkflowState) -> dict:
    next_iteration = state.get("iterations", 0) + 1

    if next_iteration == 1:
        draft = "你可以参考帮助文档处理这个问题。"
    else:
        draft = "请进入账户设置，点击安全中心，选择重置密码，并使用邮箱验证码完成确认。"

    return {
        "draft": draft,
        "iterations": next_iteration,
    }


def evaluate_response(state: ReplyWorkflowState) -> dict:
    draft = state["draft"]

    if "账户设置" in draft and "邮箱验证码" in draft:
        return {
            "quality": "pass",
            "feedback": "回复包含明确步骤，可以发送。",
        }

    return {
        "quality": "revise",
        "feedback": "回复太泛，需要补充具体步骤。",
    }


def choose_after_evaluation(state: ReplyWorkflowState) -> Literal["draft_response", "finalize_response"]:
    if state["quality"] == "pass":
        return "finalize_response"
    return "draft_response"


def finalize_response(state: ReplyWorkflowState) -> dict:
    return {
        "final_answer": state["draft"],
    }


## 4. 组装 Workflow Graph

这是一个稳定流程。

节点顺序和重试条件都是系统定义的。

In [69]:
reply_workflow_builder = StateGraph(ReplyWorkflowState)

reply_workflow_builder.add_node("draft_response", draft_response)
reply_workflow_builder.add_node("evaluate_response", evaluate_response)
reply_workflow_builder.add_node("finalize_response", finalize_response)

reply_workflow_builder.add_edge(START, "draft_response")
reply_workflow_builder.add_edge("draft_response", "evaluate_response")
reply_workflow_builder.add_conditional_edges(
    "evaluate_response",
    choose_after_evaluation,
    ["draft_response", "finalize_response"],
)
reply_workflow_builder.add_edge("finalize_response", END)

reply_workflow = reply_workflow_builder.compile()

workflow_result = reply_workflow.invoke(
    {
        "user_request": "用户想知道怎么重置密码",
        "draft": "",
        "feedback": "",
        "quality": "",
        "iterations": 0,
        "final_answer": "",
    }
)

print("iterations:", workflow_result["iterations"])
print("feedback:", workflow_result["feedback"])
print("final_answer:", workflow_result["final_answer"])

iterations: 2
feedback: 回复包含明确步骤，可以发送。
final_answer: 请进入账户设置，点击安全中心，选择重置密码，并使用邮箱验证码完成确认。


## 5. Agent 示例：动态选择工具

Agent 的特点是：

```text
模型根据用户问题决定是否调用工具、调用哪个工具、是否继续。
```

这里仍然用 fake model。

它会根据用户消息选择：

- 普通产品问题：调用 `search_policy`
- bug 问题：调用 `open_ticket`

虽然是 fake model，但 graph 结构和真实 agent loop 一样。

In [70]:
@tool
def search_policy(query: str) -> str:
    """Search product policy documents."""
    return "帮助文档：重置密码需要邮箱验证码。"


@tool
def open_ticket(summary: str) -> str:
    """Open an engineering ticket."""
    return "BUG-2026-0604-002"


agent_tools = [search_policy, open_ticket]
agent_tools_by_name = {item.name: item for item in agent_tools}


class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]
    steps: int


class FakeDynamicAgentModel:
    def __init__(self):
        self.calls = 0

    def invoke(self, messages: list[AnyMessage]) -> AIMessage:
        self.calls += 1

        if self.calls == 1:
            user_text = messages[0].content.lower()
            if "bug" in user_text or "error" in user_text:
                return AIMessage(
                    content="",
                    tool_calls=[
                        {
                            "name": "open_ticket",
                            "args": {"summary": "用户反馈上传报错"},
                            "id": "call_1",
                        }
                    ],
                )

            return AIMessage(
                content="",
                tool_calls=[
                    {
                        "name": "search_policy",
                        "args": {"query": "password reset"},
                        "id": "call_1",
                    }
                ],
            )

        last_tool_message = messages[-1].content
        return AIMessage(content="基于工具结果回答：" + last_tool_message)


## 6. 组装 Agent Graph

Agent graph 的结构和 Quickstart 类似：

```text
llm_call -> tool_node -> llm_call -> END
```

区别是：模型会根据问题选择不同工具。

In [71]:
def build_dynamic_agent():
    model = FakeDynamicAgentModel()

    def llm_call(state: AgentState) -> dict:
        return {
            "messages": [model.invoke(state["messages"])],
            "steps": state.get("steps", 0) + 1,
        }

    def tool_node(state: AgentState) -> dict:
        results = []
        last_message = state["messages"][-1]

        for tool_call in last_message.tool_calls:
            selected_tool = agent_tools_by_name[tool_call["name"]]
            observation = selected_tool.invoke(tool_call["args"])
            results.append(
                ToolMessage(
                    content=str(observation),
                    tool_call_id=tool_call["id"],
                )
            )

        return {"messages": results}

    def should_continue(state: AgentState) -> Literal["tool_node", "__end__"]:
        last_message = state["messages"][-1]
        if getattr(last_message, "tool_calls", None):
            return "tool_node"
        return END

    builder = StateGraph(AgentState)
    builder.add_node("llm_call", llm_call)
    builder.add_node("tool_node", tool_node)
    builder.add_edge(START, "llm_call")
    builder.add_conditional_edges("llm_call", should_continue, ["tool_node", END])
    builder.add_edge("tool_node", "llm_call")

    return builder.compile()


## 7. 执行 Agent Graph

先跑产品问题，再跑 bug 问题。

你会看到 agent 选择了不同工具。

In [72]:
def run_agent_case(user_message: str) -> None:
    agent = build_dynamic_agent()
    result = agent.invoke(
        {
            "messages": [HumanMessage(content=user_message)],
            "steps": 0,
        }
    )

    print("user:", user_message)
    for message in result["messages"]:
        print(type(message).__name__, "=>", getattr(message, "content", ""))
        tool_calls = getattr(message, "tool_calls", None)
        if tool_calls:
            print("tool_calls:", tool_calls)
    print("steps:", result["steps"])


run_agent_case("How do I reset my password?")
print("---")
run_agent_case("There is a bug when I upload a file.")

user: How do I reset my password?
HumanMessage => How do I reset my password?
AIMessage => 
tool_calls: [{'name': 'search_policy', 'args': {'query': 'password reset'}, 'id': 'call_1', 'type': 'tool_call'}]
ToolMessage => 帮助文档：重置密码需要邮箱验证码。
AIMessage => 基于工具结果回答：帮助文档：重置密码需要邮箱验证码。
steps: 2
---
user: There is a bug when I upload a file.
HumanMessage => There is a bug when I upload a file.
AIMessage => 
tool_calls: [{'name': 'open_ticket', 'args': {'summary': '用户反馈上传报错'}, 'id': 'call_1', 'type': 'tool_call'}]
ToolMessage => BUG-2026-0604-002
AIMessage => 基于工具结果回答：BUG-2026-0604-002
steps: 2


## 8. 什么时候用 Workflow，什么时候用 Agent

| 问题 | 更适合 |
| --- | --- |
| 步骤稳定、验收标准明确 | Workflow |
| 需要严格审批和验证 | Workflow |
| 需要模型探索未知路径 | Agent |
| 用户问题开放，工具选择不固定 | Agent |
| 既有稳定主流程，又有局部开放探索 | Workflow + Agent node |

不要把所有事情都做成 agent。

更可靠的做法通常是：

```text
外层 workflow 控制边界，
局部节点里使用 agent 处理开放问题。
```

## 9. 和 Harness 的关系

本仓库 Harness 里的 Query Loop 更接近 agent runtime：

```text
planner -> action -> tool/delegate/answer -> continue or stop
```

但 Harness 里的很多控制点可以 workflow 化：

- approval 是固定门禁
- verification 是固定门禁
- synthesis 是固定收口
- subagent research 可以是 worker 节点

所以你可以这样理解：

```text
Workflow 负责稳定边界。
Agent 负责开放决策。
Harness 负责应用层安全和恢复。
```

## 10. 本讲练习

请判断下面场景更适合 workflow、agent，还是 workflow + agent node：

1. 报销单 OCR、金额校验、主管审批、入账。
2. 用户问一个开放研究问题，需要自己决定查哪些资料。
3. 代码改造必须先 research，再 implementation，再 verification。
4. 客服流程固定，但其中“查找相关政策”可能需要多个工具探索。

参考答案：

1. workflow
2. agent
3. workflow
4. workflow + agent node

## 11. 本讲小结

这一讲的核心：

```text
Workflow 是系统控制流程，Agent 是模型控制下一步。
```

你现在应该能判断：

- workflow 和 agent 的控制权差异
- evaluator-optimizer workflow 怎么跑
- agent loop 如何动态选择工具
- 为什么很多真实系统应该用 workflow 包住 agent
- Harness 的哪些部分可以映射成 workflow 节点

下一讲可以继续学习 LangGraph 的 Workflows。